## Homework 2

> [!NOTE]
> This homework uses the pinned 2026 car fuel-efficiency release in the course
> repository. The plan and report are available in `cohorts/2026/data/`.

### Dataset

For this homework, we'll use the 2026 Car Fuel Efficiency dataset. Download it from <a href='https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'>here</a>.

You can do it with wget:
```bash
wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv
```

The goal of this homework is to create a regression model for predicting the car fuel efficiency (column `'fuel_efficiency_mpg'`).

### Preparing the dataset 

Use only the following columns:

* `'engine_displacement'`,
* `'horsepower'`,
* `'vehicle_weight'`,
* `'model_year'`,
* `'fuel_efficiency_mpg'`

### EDA

* Look at the `fuel_efficiency_mpg` variable. Does it have a long tail? 

In [4]:
import pandas as pd
import numpy as np

In [7]:
data = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'
!wget $data 

--2026-10-04 15:05:29--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 635180 (620K) [text/plain]
Saving to: ‘car_fuel_efficiency_2026.csv.1’

car_fuel_efficiency 100%[===================>] 620.29K  --.-KB/s    in 0.04s   

2026-10-04 15:05:29 (15.2 MB/s) - ‘car_fuel_efficiency_2026.csv.1’ saved [635180/635180]



In [9]:
df = pd.read_csv('car_fuel_efficiency_2026.csv')
df.dtypes

model_year               int64
origin                  object
fuel_type               object
drivetrain              object
num_doors                int64
engine_displacement      int64
num_cylinders            int64
horsepower             float64
vehicle_weight           int64
acceleration           float64
fuel_efficiency_mpg    float64
dtype: object

## Question 1

There's one column with missing values. What is it?

* `'engine_displacement'`
* `'horsepower'`
* `'vehicle_weight'`
* `'model_year'`

In [11]:
colums_list = ['engine_displacement','horsepower','vehicle_weight','model_year','fuel_efficiency_mpg']
df[colums_list].isnull().sum()

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64

## Question 2

What's the median (50% percentile) for variable `'horsepower'`?

- 204
- 254
- 304
- 354

In [68]:
df['horsepower'].median()

254.0

### Prepare and split the dataset

Shuffle the filtered dataset and create the split exactly as in the lecture:

```python
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]
```

For Q5, repeat the same block with each listed seed. For Q6, use seed `9`.


## Question 3

* We need to deal with missing values for the column from Q1.
* We have two options: fill it with 0 or with the mean of this variable.
* Try both options. For each, train a linear regression model without regularization using the code from the lessons.
* For computing the mean, use the training only!
* Use the validation dataset to evaluate the models and compare the RMSE of each option.
* Round the RMSE scores to 3 decimal digits using `round(score, 3)`. This
  keeps the imputation difference visible in this release.
* Which option gives better RMSE?

Options:

- With 0
- With mean
- Both are equally good

In [69]:
features = [
    'engine_displacement',
    'horsepower',
    'vehicle_weight',
    'model_year'
]

target = 'fuel_efficiency_mpg'

In [18]:
n = len(df)
n

10000

In [70]:
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

n_val, n_test, n_train

(2000, 2000, 6000)

In [71]:
np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

In [73]:
df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

In [74]:
y_train = df_train[target].values
y_val = df_val[target].values
y_test = df_test[target].values

In [75]:
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)
    
    return w_full[0], w_full[1:]

In [76]:
def rmse(y, y_pred):
    se = (y - y_pred) ** 2
    mse = se.mean()
    return np.sqrt(mse)

### Option1: fill it with 0

In [80]:
X_train_zero = df_train[features].fillna(0).values
X_val_zero = df_val[features].fillna(0).values

w0_zero, w_zero = train_linear_regression(X_train_zero, y_train)

y_pred_zero = w0_zero + X_val_zero.dot(w_zero)
score_zero = rmse(y_val, y_pred_zero)

round(score_zero, 3)

np.float64(2.183)

### Option2: it with mean

In [83]:
mean_hp = df_train['horsepower'].mean()

X_train_mean = df_train[features].fillna({
    'horsepower': mean_hp
}).values

X_val_mean = df_val[features].fillna({
    'horsepower': mean_hp
}).values

w0_mean, w_mean = train_linear_regression(X_train_mean, y_train)

y_pred_mean = w0_mean + X_val_mean.dot(w_mean)
score_mean = rmse(y_val, y_pred_mean)

round(score_mean, 3)

np.float64(2.181)


## Question 4

* Now let's train a regularized linear regression.
* For this question, fill the NAs with 0. 
* Try different values of `r` from this list: `[0, 0.01, 0.1, 1, 5, 10, 100]`.
* Use RMSE to evaluate the model on the validation dataset.
* Round the RMSE scores to 4 decimal digits. This keeps the small but real
  regularization differences visible instead of turning several choices into a tie.
* Which `r` gives the best RMSE?

If multiple options give the same best RMSE, select the smallest `r`.

Options:

- 0
- 0.01
- 0.1
- 1
- 5
- 10
- 100

In [108]:
def train_linear_regression_reg(X, y, r=0.0):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)

    R = r * np.eye(XTX.shape[0])
    R[0, 0] = 0

    XTX_inv = np.linalg.inv(XTX + R)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]

In [102]:
def prepare_X(df):
    return df[features].fillna(0).values

In [111]:
X_train = prepare_X(df_train)
X_val = prepare_X(df_val)

(6000, 4)
(2000, 4)


In [119]:
r_values = [0, 0.01, 0.1, 1, 5, 10, 100]
scores = {}

for r in r_values:
    w0, w = train_linear_regression_reg(
        X_train,
        y_train,
        r=r
    )

    y_pred = w0 + X_val.dot(w)
    score = rmse(y_val, y_pred)

    scores[r] = round(score, 4)
    print(f'r={r:<6} RMSE={scores[r]:.4f}')

best_r = min(scores, key=lambda r: (scores[r], r))

print('Best r:', best_r)
print('Best RMSE:', scores[best_r])

r=0      RMSE=2.1827
r=0.01   RMSE=2.1827
r=0.1    RMSE=2.1827
r=1      RMSE=2.1827
r=5      RMSE=2.1827
r=10     RMSE=2.1827
r=100    RMSE=2.1827
Best r: 0
Best RMSE: 2.1827


## Question 5 

* We used seed 42 for splitting the data. Let's find out how selecting the seed influences our score.
* Try different seed values: `[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]`.
* For each seed, do the train/validation/test split with 60%/20%/20% distribution.
* Fill the missing values with 0 and train a model without regularization.
* For each seed, evaluate the model on the validation dataset and collect the RMSE scores. 
* What's the standard deviation of all the scores? To compute the standard deviation, use `np.std`.
* Round the result to 3 decimal digits (`round(std, 3)`)

What's the value of std?

- 0.006
- 0.016
- 0.029
- 0.036

> Note: Standard deviation shows how different the values are.
> If it's low, then all values are approximately the same.
> If it's high, the values are different. 
> If standard deviation of scores is low, then our model is *stable*.

In [128]:
seeds = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
scores = []

for seed in seeds:
    # ต้องสร้าง idx ใหม่ทุก seed
    idx = np.arange(n)

    np.random.seed(seed)
    np.random.shuffle(idx)

    # Split 60% / 20% / 20%
    df_train = df.iloc[idx[:n_train]]
    df_val = df.iloc[idx[n_train:n_train + n_val]]
    df_test = df.iloc[idx[n_train + n_val:]]

    # Target
    y_train = df_train[target].values
    y_val = df_val[target].values

    # เติม missing values ด้วย 0
    X_train = df_train[features].fillna(0).values
    X_val = df_val[features].fillna(0).values

    # ไม่มี regularization
    w0, w = train_linear_regression(X_train, y_train)

    y_pred = w0 + X_val.dot(w)
    score = rmse(y_val, y_pred)

    scores.append(score)
    print(f'seed={seed}: RMSE={score:.4f}')

seed=0: RMSE=2.2392
seed=1: RMSE=2.2021
seed=2: RMSE=2.1634
seed=3: RMSE=2.2044
seed=4: RMSE=2.2019
seed=5: RMSE=2.2458
seed=6: RMSE=2.2697
seed=7: RMSE=2.1908
seed=8: RMSE=2.2166
seed=9: RMSE=2.2070


In [129]:
std = np.std(scores)

print('Scores:', scores)
print('Standard deviation:', std)
print('Rounded:', round(std, 3))

Scores: [np.float64(2.239204143046126), np.float64(2.2021128210838845), np.float64(2.1634197787530947), np.float64(2.2043588768539224), np.float64(2.2018800943311554), np.float64(2.2457851509085134), np.float64(2.2697212079524043), np.float64(2.190794599933443), np.float64(2.216611201686444), np.float64(2.2070365928178957)]
Standard deviation: 0.028781274078191175
Rounded: 0.029


## Question 6

* Split the dataset like previously, use seed 9.
* Combine train and validation datasets.
* Fill the missing values with 0 and train a model with `r=0.001`. 
* What's the RMSE on the test dataset?

Options:

- 0.236
- 2.236
- 22.10
- 221.0

In [136]:
idx = np.arange(n)

np.random.seed(9)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

df_train_full = pd.concat([df_train, df_val])

X_train_full = df_train_full[features].fillna(0).values
X_test = df_test[features].fillna(0).values

y_train_full = df_train_full[target].values
y_test = df_test[target].values


w0, w = train_linear_regression_reg(
    X_train,
    y_train,
    r= 0.001
)

y_pred = w0 + X_test.dot(w)

score = rmse(y_test, y_pred)
round(score, 3)

np.float64(2.236)